# Comparación v2: transferencia fija y etanol inicial de LAB016–018

Se fija **kLa = 0.07 h⁻¹ en los modelos sin/con transportadores**. Es el coeficiente publicado por Zenteno, pero se conservan nuestras ecuaciones de desgasificación: esto no es una reproducción literal de su balance. Se reajusta sólo la ganancia de cada modelo nuevo, con el mismo objetivo de medianas horarias y ponderación igual por LAB de 04_compare_models. No se ajusta la química. El histórico conserva sus coeficientes guardados por partición: su parámetro de liberación no se sustituye por kLa.

Las figuras principales usan **E0 = 9.8 g/L para LAB016–018**, en las tres formulaciones. LAB019–021 conserva sus iniciales medidas. Una sección aparte compara E0=0 y E0=9.8 con kLa fijo para separar ambos cambios. Se prueban las dos particiones de gas: calibración LAB019–021 / holdout LAB016–018, y la dirección inversa. El supuesto E0=9.8 es una sensibilidad solicitada, no una nueva medición.

El resto de entradas se conserva: temperatura del log procesada 20/45 min, dosis y volumen remanente de 04, Cd0=0 en LAB016–018 y Cd0=0.907/0.730/0.829 g/L en LAB019–021 (021 provisional), solubilidad provisional y predicciones sin filtrar. Las observaciones gaseosas son medianas horarias de RAW convertida, dibujadas como líneas. El objetivo temporal y tratamiento de valores bajos son los originales de 04: MSE de todos los puntos de su cobertura, sin introducir la política del diagnóstico EKF.

G+F conserva el ensayo conjunto; YAN incluye NUT. La química de los modelos nuevos se calibró en LAB004–009; LAB019–021 sigue reservado para química aunque participe en calibración de gas. Todas las ganancias nuevas existen sólo en memoria: no se escriben configs, results, CSV o PNG.

**Adaptación:** estas formulaciones no contienen una fase fisiológica de adaptación inicial. Sin transportadores, el incremento de actividad por YAN es instantáneo; con transportadores hay dinámica de síntesis, pero también actividad basal inmediata. Esta v2 prueba únicamente transferencia y etanol inicial, no añade una fase de adaptación.

In [ ]:
from pathlib import Path
import sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
assert Path(sys.prefix).name.lower()=='fermentation'

import hashlib
def protected_hashes():
    return {str(p):hashlib.sha256(p.read_bytes()).hexdigest() for folder in ("src","configs","results","data/compatibility") for p in (ROOT/folder).rglob("*") if p.is_file() and "__pycache__" not in p.parts}
PROTECTED=protected_hashes()


In [ ]:
from wine_fermentation import comparison as C
from wine_fermentation.comparison import PRIMARY_TRAINING,chemical_observations
from wine_fermentation.shared.simulation import simulate
from IPython.display import Markdown
from dataclasses import replace
from copy import copy
FIXED_KLA=0.07
prepared0=C.prepare(ROOT)
# Referencia de 04 sin optimizar y sin escribir sus ajustes.
reference_experiments={key:C.run(ROOT,False,key,prepared0) for key in C.PARTITIONS}
prepared98={**prepared0,'chemical':prepared0['chemical'].copy(),'contexts':{k:v.copy() for k,v in prepared0['contexts'].items()}}
v=copy(prepared0['V']);v.batches=prepared0['V'].batches.copy();v.cache=prepared0['V'].cache.copy();v.cores=prepared0['V'].cores.copy();v.theta_by_case=prepared0['V'].theta_by_case.copy()
prepared98['V']=v
for lab in C.LABS[:3]:
    b=v.batches[lab];v.batches[lab]=replace(b,initials={**b.initials,'E':9.8})
    v.cache.pop(('theta_nv2',lab),None);v.cores.pop(('theta_nv2',lab),None)
v.prepare('theta_nv2',v.thetas['theta_nv2'],C.LABS[:3])
for lab in C.LABS[:3]:prepared98['chemical']['historical',lab]=v.cores['theta_nv2',lab]
for name in C.NAMES[1:]:
    for lab in C.LABS[:3]:
        b,p,core,volume=prepared0['contexts'][name][lab]
        b=replace(b,initials={**b.initials,'E':9.8});core=simulate(name,b,p)
        prepared98['contexts'][name][lab]=(b,p,core,volume)
        prepared98['chemical'][name,lab]=core
# Entradas y trayectorias LAB019–021 no cambian al ensayar el E0 de LAB016–018.
for name in C.NAMES:
    for lab in C.LABS[3:]:
        assert prepared98['chemical'][name,lab].equals(prepared0['chemical'][name,lab])

def fixed_run(prepared,training):
    train,holdout=C.partition(training);curves={};fits={};metrics=[]
    for name in C.NAMES:
        predict=C.predictor(name,prepared)
        if name=='historical':
            fit=reference_experiments[training]['gas_fits'][name].copy()
            fit['policy']='Referencia histórica: coeficientes guardados sin reajuste'
            frames={lab:predict(lab,[fit[k] for k in C.GAS_NAMES]) for lab in C.LABS}
        else:
            unit={lab:predict(lab,[FIXED_KLA,1.]) for lab in C.LABS}
            numerator=0.;denominator=0.
            for lab in train:
                f=unit[lab];o=prepared['observations'][lab]
                o=o[o.time_h.between(f.index.min(),f.index.max())]
                assert len(o)>0
                a=np.interp(o.time_h,f.index,f.qpred);b=o.co2_raw_g_l_h.to_numpy()
                numerator+=np.mean(a*b);denominator+=np.mean(a*a)
            gain=float(np.clip(numerator/denominator,.05,3.))
            fit=dict(kLa=FIXED_KLA,gain=gain,training_labs=list(train),holdout_labs=list(holdout),
                     gain_at_bound=bool(gain in (.05,3.)),policy='kLa fijo; ganancia por mínimos cuadrados con peso igual por LAB')
            frames={}
            for lab,f in unit.items():
                f=f.copy();f['qpred']=gain*f.qgas;frames[lab]=f
                balance=np.max(abs(np.diff(f.Cd)-np.diff(f.index)*(f.qprod.to_numpy()[:-1]-f.qgas.to_numpy()[1:])))
                assert balance<1e-8
        fits[name]=fit
        for lab,f in frames.items():
            curves[name,lab]=f;o=prepared['observations'][lab]
            o=o[o.time_h.between(f.index.min(),f.index.max())]
            error=np.interp(o.time_h,f.index,f.qpred)-o.co2_raw_g_l_h.to_numpy()
            metrics.append(dict(model=name,training=training,lab=lab,n=len(o),split='gas_training' if lab in train else 'gas_validation',RMSE=float(np.sqrt(np.mean(error**2))),bias=float(error.mean())))
    return dict(curves=curves,chemical=prepared['chemical'],metrics=pd.DataFrame(metrics),
                chemical_metrics=reference_experiments[training]['chemical_metrics'].copy(),gas_fits=fits,
                data=prepared['data'],V=prepared['V'],training=training,training_labs=train,holdout_labs=holdout)
experiments0={key:fixed_run(prepared0,key) for key in C.PARTITIONS}
experiments={key:fixed_run(prepared98,key) for key in C.PARTITIONS}
result=experiments[PRIMARY_TRAINING]
combined=pd.concat([res['metrics'] for res in experiments.values()],ignore_index=True)
for key,res in experiments.items():
    display(Markdown('## CO₂ '+('principal' if key==PRIMARY_TRAINING else 'inverso')+': '+key))
    display(res['metrics'])
    display(pd.DataFrame(res['gas_fits']).T)
display(Markdown('## Media de RMSE por fermentación [g/(L·h)] · calibración y holdout separados'))
display(combined.groupby(['model','training','split']).RMSE.mean().unstack(['training','split']))
paired=combined.pivot(index=['model','lab'],columns='training',values='RMSE')
paired['delta_RMSE_principal_menos_inversa']=paired['train_019_021']-paired['train_016_018']
paired['rol_principal']=['Calibración CO₂' if lab in result['training_labs'] else 'Holdout CO₂' for _,lab in paired.index]
display(Markdown('## Mismo LAB, dos ajustes · delta negativo = menor RMSE con el ajuste principal; el rol del LAB cambia'))
display(paired)
display(Markdown('## Holdout químico LAB019–021 · parámetros químicos idénticos en ambos ensayos de CO₂'))
display(result['chemical_metrics'])

sample_checks=[]
for lab in ('LAB019','LAB020','LAB021'):
    for state in ('GF','N'):
        plotted=chemical_observations(result['V'].chem,lab,state)
        scored=chemical_observations(result['V'].chem,lab,state,after_initial=True)
        col='GF_g_l' if state=='GF' else 'N_kg_m3'
        source=result['V'].chem.loc[result['V'].chem.lab.eq(lab)]
        assert len(plotted)==int(source[col].notna().sum())
        measured_nut=source.loc[source.is_NUT & source[col].notna(),'sample']
        assert set(measured_nut).issubset(set(plotted['sample']))
        for model in ('historical','without_transporters','with_transporters'):
            row=result['chemical_metrics'].query('model==@model and lab==@lab and state==@state')
            assert len(row)==1 and int(row.n.iloc[0])==len(scored)
        sample_checks.append(dict(lab=lab,variable=state,observaciones_en_figura=len(plotted),
            observaciones_en_metricas=len(scored),muestras_NUT_incluidas=len(measured_nut)))
display(Markdown('## Cobertura de las muestras de G+F y YAN'))
display(pd.DataFrame(sample_checks))


In [ ]:
from wine_fermentation.comparison import observations
calibration_observations=observations(result['data'])
colors={'historical':'#0072B2','without_transporters':'#D55E00','with_transporters':'#009E73'}
names={'historical':'Histórico · coeficientes previos','without_transporters':'Sin transportadores','with_transporters':'Con transportadores'}
for key,res in experiments.items():
    ordered_labs=res['training_labs']+res['holdout_labs']
    train_label=', '.join(res['training_labs']);holdout_label=', '.join(res['holdout_labs'])
    fig,axes=plt.subplots(2,3,figsize=(15,9))
    fig.subplots_adjust(top=.77,bottom=.08,hspace=.43,wspace=.26)
    fig.suptitle('CO₂ emitido · kLa fijo 0.07 h⁻¹ · E₀ LAB016–018 = 9.8 g/L'+'\nCalibración: '+train_label+' | Holdout: '+holdout_label,y=.99,fontsize=14)
    for ax,lab in zip(axes.flat,ordered_labs):
        obs=calibration_observations[lab]
        f=res['curves']['historical',lab]
        obs=obs[obs.time_h.between(f.index.min(),f.index.max())]
        ax.plot(obs.time_h,obs.co2_raw_g_l_h,color='black',lw=1.5,label='Medianas horarias del sensor',zorder=4)
        for name in colors:
            f=res['curves'][name,lab];ax.plot(f.index,f.qpred,color=colors[name],label=names[name])
        split='Calibración CO₂' if lab in res['training_labs'] else 'Holdout CO₂ · sin reajuste'
        ax.set_title(lab+' · '+split,fontsize=11)
        ax.set_ylim(bottom=0);ax.margins(y=.06)
        ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ emitido [g/(L·h)]');ax.grid(alpha=.25)
    handles,labels=axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.90),ncol=4,title='Datos y formulaciones · química fija, modelos nuevos: sólo ganancia ajustada con la fila superior')
    plt.show()

chemical_names={**names,'historical':'Histórico · theta_nv2 fijo'}


## Glucosa + fructosa
Se usa la medición conjunta disponible de G+F; no se exige disponer de glucosa y fructosa individuales ni se imputan resultados ausentes. Las curvas predichas son G+F del modelo. Primero se muestran los ensayos del ajuste químico y después el holdout, sin recalibrar.

In [ ]:
from wine_fermentation.data import natural_batches
from wine_fermentation.shared.simulation import simulate
natural_observations,natural_training_batches,_=natural_batches(ROOT)
chemical_parameters={'historical':json.loads((ROOT/'configs/theta/theta_nv2.json').read_text(encoding='utf-8'))}
for name in ('without_transporters','with_transporters'):
    chemical_parameters[name]=json.loads((ROOT/f'configs/models/{name}/fit.json').read_text(encoding='utf-8'))['parameters']
fig,axes=plt.subplots(2,3,figsize=(15,9))
fig.subplots_adjust(top=.79,bottom=.08,hspace=.43,wspace=.26)
fig.suptitle('Glucosa + fructosa · mosto natural LAB004–009\nDatos de calibración química de los modelos nuevos · histórico congelado',y=.99,fontsize=15)
for ax,batch in zip(axes.flat,natural_training_batches):
    obs=natural_observations.loc[natural_observations.batch.eq(batch.batch)].dropna(subset=['S_GF_g_l'])
    obs=obs.loc[obs.time_h.gt(0)]
    ax.scatter(obs.time_h,obs.S_GF_g_l,color='black',s=16,label='G + F medido')
    ax.scatter([0],[batch.initials['G']+batch.initials['F']],facecolors='none',edgecolors='black',s=28,label='Condición inicial')
    for name in colors:
        f=simulate(name,batch,chemical_parameters[name])
        ax.plot(f.index,f.G+f.F,color=colors[name],label=chemical_names[name])
    ax.set_title(batch.batch+' · Calibración química',fontsize=11)
    ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('Glucosa + fructosa [g/L]');ax.grid(alpha=.25)
handles,labels=axes[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.90),ncol=3,title='Datos y formulaciones')
plt.show()


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.8))
fig.subplots_adjust(top=.60,bottom=.15,wspace=.28)
fig.suptitle('Glucosa + fructosa · predicción química en holdout LAB019–021\nParámetros congelados · sin reajuste a estos ensayos',y=.99,fontsize=14)
for ax,lab in zip(axes,('LAB019','LAB020','LAB021')):
    obs=chemical_observations(result['V'].chem,lab,'GF')
    observed=obs.loc[obs.t_h.gt(0)]
    initial=obs.loc[obs.t_h.le(0)]
    ax.scatter(observed.t_h,observed.GF_g_l,color='black',s=16,label='G + F medido')
    ax.scatter(initial.t_h,initial.GF_g_l,facecolors='none',edgecolors='black',s=28,label='Muestra inicial')
    for name in colors:
        f=result['chemical'][name,lab]
        ax.plot(f.index,f.G+f.F,color=colors[name],label=chemical_names[name])
    ax.set_title(lab+' · Holdout',fontsize=11)
    ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('Glucosa + fructosa [g/L]');ax.grid(alpha=.25)
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.85),ncol=3,title='Datos y formulaciones')
plt.show()


In [ ]:
columns={'X':'X_viable_g_l','N':'N_kg_m3','G':'G_g_l','F':'F_g_l','E':'E_g_l'}
state_names={'X':'Biomasa viable','N':'YAN','G':'Glucosa','F':'Fructosa','E':'Etanol'}
for state,col in columns.items():
    fig,axes=plt.subplots(1,3,figsize=(15,4.8))
    fig.subplots_adjust(top=.68,bottom=.15,wspace=.28)
    fig.suptitle(state_names[state]+' · predicción química en holdout LAB019–021\nParámetros congelados · sin reajuste a estos ensayos',y=.99,fontsize=14)
    for ax,lab in zip(axes,('LAB019','LAB020','LAB021')):
        obs=chemical_observations(result['V'].chem,lab,state)
        ax.scatter(obs.t_h,obs[col],color='black',s=14,label='Mediciones offline (incluye NUT)' if state=='N' else 'Mediciones offline')
        for name in colors:
            f=result['chemical'][name,lab];ax.plot(f.index,f[state],color=colors[name],label=chemical_names[name])
        ax.set_title(lab+' · Holdout',fontsize=11)
        ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel(state_names[state]+' [g/L]');ax.grid(alpha=.25)
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.85),ncol=4,title='Datos y formulaciones')
    plt.show()


In [ ]:
for key,res in experiments.items():
    fig,axes=plt.subplots(2,3,figsize=(16,10))
    fig.subplots_adjust(top=.66,bottom=.08,hspace=.44,wspace=.27)
    fig.suptitle('CO₂ disuelto y transferencia · kLa = 0.07 h⁻¹'+'\nLAB019–021: '+('calibración CO₂' if key==PRIMARY_TRAINING else 'holdout CO₂')+' | Solubilidad PROVISIONAL | Antes de ganancia',y=.99,fontsize=14)
    for j,lab in enumerate(('LAB019','LAB020','LAB021')):
        for name in ('without_transporters','with_transporters'):
            f=res['curves'][name,lab]
            axes[0,j].plot(f.index,f.Cd,color=colors[name],label=names[name]+' · Cd')
            axes[0,j].plot(f.index,f.Csat,color=colors[name],ls=':',label=names[name]+' · Csat provisional')
            axes[1,j].plot(f.index,f.qprod,color=colors[name],ls='--',label=names[name]+' · producción')
            axes[1,j].plot(f.index,f.qgas,color=colors[name],label=names[name]+' · emisión')
        axes[0,j].scatter([0],[{'LAB019':.907,'LAB020':.730,'LAB021':.829}[lab]],color='black',s=22,label='Cd inicial medido (021 provisional)')
        axes[0,j].set_title(lab+' · '+('Calibración CO₂' if lab in res['training_labs'] else 'Holdout CO₂')+' · Depósito',fontsize=11)
        axes[1,j].set_title(lab+' · Producción y emisión (predicción)',fontsize=11)
        for ax in axes[:,j]:ax.set_xlabel('Tiempo desde inoculación [h]');ax.grid(alpha=.25)
        axes[0,j].set_ylabel('CO₂ disuelto [g/L]')
        axes[1,j].set_ylabel('Caudal de CO₂ [g/(L·h)]')
    handles,labels=axes[0,0].get_legend_handles_labels()
    h2,l2=axes[1,0].get_legend_handles_labels()
    fig.legend(handles+h2,labels+l2,loc='upper center',bbox_to_anchor=(.5,.91),ncol=3,title='Curvas calculadas: Cd = concentración disuelta; Csat = saturación | Punto: condición inicial medida',fontsize=9,title_fontsize=10)
    plt.show()


## Efectos separados: transferencia y etanol inicial
Las primeras comparaciones cambian únicamente kLa y reajustan la ganancia con E0=0. La última cambia E0 a 9.8 g/L en LAB016–018. No hay cambios en theta ni adaptación fisiológica. Las figuras siguientes mantienen sólo dos curvas del mismo modelo por panel, con E0=0 y 9.8, usando la partición de calibración LAB019–021; por ello la ganancia es idéntica en cada pareja.

In [ ]:
rows=[]
for variant,exps in [('04 original · E0=0',reference_experiments),('kLa fijo · E0=0',experiments0),('kLa fijo · E0=9.8',experiments)]:
    for key,res in exps.items():
        rows.append(res['metrics'].assign(variant=variant))
all_metrics=pd.concat(rows,ignore_index=True)
display(all_metrics.groupby(['variant','model','training','split']).RMSE.mean().unstack('split'))
for name in C.NAMES:
    fig,axes=plt.subplots(1,3,figsize=(15,4.8));fig.subplots_adjust(top=.65,bottom=.15,wspace=.28)
    fig.suptitle(names[name]+' · efecto de E₀ en LAB016–018\nCalibración CO₂: LAB019–021 | Holdout: LAB016–018',y=.99,fontsize=13)
    for ax,lab in zip(axes,C.LABS[:3]):
        o=calibration_observations[lab];ax.plot(o.time_h,o.co2_raw_g_l_h,color='black',lw=1.3,label='Medianas horarias del sensor')
        for label,exps,style in [('E₀ = 0 g/L',experiments0,'--'),('E₀ = 9.8 g/L',experiments,'-')]:
            f=exps[PRIMARY_TRAINING]['curves'][name,lab];ax.plot(f.index,f.qpred,color='#0072B2' if style=='--' else '#D55E00',ls=style,label=label)
        ax.set_title(lab+' · Holdout CO₂');ax.set_ylim(bottom=0);ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ [g/(L·h)]')
    h,l=axes[0].get_legend_handles_labels();fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.85),ncol=3,title='Mismos coeficientes de CO₂ dentro de cada pareja');plt.show()
# Producción inicial y saturación: no confundir con emisión al sensor.
rows=[]
for variant,exps in [('E0=0',experiments0),('E0=9.8',experiments)]:
    for name in C.NAMES[1:]:
        for lab in C.LABS[:3]:
            f=exps[PRIMARY_TRAINING]['curves'][name,lab]
            rows.append(dict(variant=variant,model=name,lab=lab,qprod0=f.qprod.iloc[0],Csat0=f.Csat.iloc[0],Cd0=f.Cd.iloc[0]))
display(pd.DataFrame(rows))
for state,title in [('GF','Glucosa + fructosa'),('E','Etanol')]:
    fig,axes=plt.subplots(1,3,figsize=(15,4.8));fig.subplots_adjust(top=.65,bottom=.15,wspace=.28)
    fig.suptitle(title+' · LAB016–018 · E₀ = 9.8 g/L\nPredicciones químicas; sin series Y15 de referencia en esta fuente',y=.99,fontsize=13)
    for ax,lab in zip(axes,C.LABS[:3]):
        for name in C.NAMES:
            f=result['chemical'][name,lab];y=f.G+f.F if state=='GF' else f.E
            ax.plot(f.index,y,color=colors[name],label=chemical_names[name])
        ax.set_title(lab);ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('g/L')
    h,l=axes[0].get_legend_handles_labels();fig.legend(h,l,loc='upper center',bbox_to_anchor=(.5,.85),ncol=3,title='Parámetros químicos congelados');plt.show()
assert all(res['gas_fits'][name]['kLa']==FIXED_KLA for res in experiments.values() for name in C.NAMES[1:])
assert all(res['chemical']['historical',lab].E.iloc[0]==9.8 for res in experiments.values() for lab in C.LABS[:3])
assert PROTECTED==protected_hashes(),'Se modificaron modelos, parámetros o resultados guardados'
print('Verificado: kLa fijo, particiones separadas, balances, E0 ensayado y archivos protegidos intactos. Sin CSV/PNG exportados.')
